In [ ]:
# uncomment these and run this cell if needed
!pip install evaluate
!pip install huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.6 MB/s eta 0:00:00


In [ ]:
from datasets import load_dataset, Dataset
from transformers import AutoModelForSequenceClassification
from transformers import AutoTokenizer
from transformers import get_scheduler
from transformers import Trainer, TrainingArguments
from datasets import concatenate_datasets
import evaluate
import numpy as np
import requests
import time

In [ ]:
#initalize variables
header1 = "Passage: "
header2 = "Question: "
header3 = "Answer: "

column_names1 = "citing_prompt"
column_names2 = "question"
column_names3 = "holding_1"

def set_header1(header_value):
  global header1
  header1=header_value

def set_header2(header_value):
  global header2
  header2=header_value

def set_header3(header_value):
  global header3
  header3=header_value

def set_column_names1(column_names):
  global column_names1
  column_names1=column_names

def set_column_names2(column_names):
  global column_names2
  column_names2=column_names

def set_column_names3(column_names):
  global column_names3
  column_names3=column_names

def set_master_columns(col1, col2, col3, header_val1, header_val2, header_val3):
  global column_names1, column_names2, column_names3, header1, header2, header3
  column_names1=col1
  column_names2=col2
  column_names3=col3

  header1=header_val1
  header2=header_val2
  header3=header_val3

In [ ]:
# prepping the dataset

#ds = load_dataset("casehold/casehold", "fold_1") does not work due to the dataset script being out of date
url = "https://datasets-server.huggingface.co/rows"

all_rows = []
offset = 0
batch_size = 100



while len(all_rows) < 2000:
    time.sleep(1)
    params = {
        "dataset": "casehold/casehold",
        "config": "fold_1",
        "split": "train",
        "offset": offset,
        "length": batch_size
    }

    response = requests.get(url, params=params)
    try:
        data = response.json()
    except:
        time.sleep(10)

    batch = [row["row"] for row in data["rows"]]

    if not batch:  # no more data
        break

    all_rows.extend(batch)
    offset += batch_size


#Convert list of rows into Hugging Face Dataset
raw_datasets_legal = Dataset.from_list(all_rows) #https://huggingface.co/datasets/casehold/casehold/viewer/fold_1

def label_map(example):
    if example["label"] == "PASS" or example["label"] == "true" or example["label"] == "1" or example["label"] == 1:
        example["label"] = 1
    else:
        example["label"] = 0
    return example

raw_datasets_legal= raw_datasets_legal.map(label_map)

#convert legal columns
def convert_legal(example):
    return {
        "citing_prompt": example["citing_prompt"],
        "question": "What is a true legal fact?",
        "holding_1": example["holding_1"],
        "label": example["label"]
    }
raw_datasets_legal = raw_datasets_legal.map(convert_legal)




Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [ ]:
from datasets import Value
from datasets import DatasetDict
cols = ["citing_prompt", "question", "holding_1", "label"]


legal_clean = raw_datasets_legal.remove_columns(
    [c for c in raw_datasets_legal.column_names if c not in cols]
)

legal_clean = legal_clean.cast_column("label", Value("int64"))



Casting the dataset:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [ ]:
label_0 = legal_clean.filter(lambda x: x["label"] == 0)
label_1 = legal_clean.filter(lambda x: x["label"] == 1)

min_size = min(len(label_0), len(label_1))

label_0 = label_0.shuffle(seed=42).select(range(min_size))
label_1 = label_1.shuffle(seed=42).select(range(min_size))

balanced_set = concatenate_datasets([label_0, label_1]).shuffle(seed=42)

Filter:   0%|          | 0/2000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [ ]:
split_datasets = balanced_set.train_test_split(test_size=0.2, seed=42)

In [ ]:
small_train_dataset = split_datasets["train"].shuffle(seed=42).select(range(400))
small_eval_dataset = split_datasets["test"].shuffle(seed=42).select(range(100))

In [ ]:
rag_docs = load_dataset("isaacus/legal-rag-bench", "corpus") #https://huggingface.co/datasets/isaacus/legal-rag-bench?library=datasets

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

corpus.jsonl: 0.00B [00:00, ?B/s]

Generating test split:   0%|          | 0/4876 [00:00<?, ? examples/s]

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
url = "https://datasets-server.huggingface.co/rows"

all_rows = []
offset = 0
batch_size = 100



while len(all_rows) < 15000:
    time.sleep(1)
    params = {
        "dataset": "casehold/casehold",
        "config": "fold_2",
        "split": "train",
        "offset": offset,
        "length": batch_size
    }

    response = requests.get(url, params=params)
    try:
        data = response.json()
    except:
        time.sleep(10)

    batch = [row["row"] for row in data["rows"]]

    if not batch:  # no more data
        break

    all_rows.extend(batch)
    offset += batch_size


# Convert list of rows into Hugging Face Dataset
rag_docs = Dataset.from_list(all_rows) #https://huggingface.co/datasets/casehold/casehold/viewer/fold_2


In [ ]:
#knowledge_base = small_train_dataset["citing_prompt"]

"""
def add_rag_context(example):
    query = str(example["citing_prompt"])+""
    query_words = set(query.lower().split())

    best_passage = ""
    best_score = 0

    current_passage = str(example["citing_prompt"])
    for passage in knowledge_base:
        passage_text = str(passage)

        #skip itself
        if current_passage in passage_text:
            best_passage = passage_text
            break

        passage_words = set(passage_text.lower().split())
        score = len(query_words.intersection(passage_words))

        if score > best_score:
            best_score = score
            best_passage = passage_text

    example["rag_context"] = best_passage
    return example
"""

'\ndef add_rag_context(example):\n    query = str(example["citing_prompt"])+""\n    query_words = set(query.lower().split())\n\n    best_passage = ""\n    best_score = 0\n\n    current_passage = str(example["citing_prompt"])\n    for passage in knowledge_base:\n        passage_text = str(passage)\n\n        #skip itself\n        if current_passage in passage_text:\n            best_passage = passage_text\n            break\n\n        passage_words = set(passage_text.lower().split())\n        score = len(query_words.intersection(passage_words))\n\n        if score > best_score:\n            best_score = score\n            best_passage = passage_text\n\n    example["rag_context"] = best_passage\n    return example\n'

In [ ]:
!pip install sentence-transformers
from sentence_transformers import SentenceTransformer, util
import torch

In [ ]:
def chunk_text(text, max_tokens=100):
    token_ids = tokenizer.encode(str(text), add_special_tokens=False)

    return [token_ids[i:i + max_tokens] for i in range(0, len(token_ids), max_tokens)]

In [ ]:
knowledge_base_tokens  = []

for context in rag_docs["citing_prompt"]:
    knowledge_base_tokens.extend(chunk_text(context, max_tokens=100))

In [ ]:
knowledge_base = [
    tokenizer.decode(chunk, skip_special_tokens=True)
    for chunk in knowledge_base_tokens
]

In [ ]:
len(knowledge_base)

43588

In [ ]:
embedder = SentenceTransformer("AdamLucek/ModernBERT-embed-base-legal-MRL")


kb_embeddings = embedder.encode(
    knowledge_base,
    convert_to_tensor=True,
    show_progress_bar=True,
    batch_size=64
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/682 [00:00<?, ?it/s]

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


#Save embeddings
torch.save(kb_embeddings.cpu(), "/content/drive/MyDrive/kb_Legal_embeddings_2.pt")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

kb_embeddings = torch.load("/content/drive/MyDrive/kb_Legal_embeddings_2.pt")
kb_embeddings = kb_embeddings.to(device)

In [ ]:
def add_rag_context(example):
    query = str(example["citing_prompt"]).strip()

    if query == "":
        query = str(example["holding_1"]).strip()

    query_embedding = embedder.encode(
        query,
        convert_to_tensor=True,
        show_progress_bar=False
    )

    scores = util.cos_sim(query_embedding, kb_embeddings)

    #take top 3 from the single row
    top_indices = torch.topk(scores[0], k=3).indices

    combined = "\n".join(knowledge_base[idx] for idx in top_indices)

    #get token count
    token_ids = tokenizer.encode(combined, add_special_tokens=False)
    token_count = len(token_ids)

    example["rag_context"] = combined
    return example

In [ ]:
"""
#Embedding style
!pip install -q sentence-transformers faiss-cpu
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

embedder = SentenceTransformer("all-MiniLM-L6-v2")

knowledge_base_texts = [str(x) for x in knowledge_base]

kb_embeddings = embedder.encode(
    knowledge_base_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

index = faiss.IndexFlatIP(kb_embeddings.shape[1])
index.add(kb_embeddings)

def add_rag_context(example):
    query = str(example["citing_prompt"]) + " " + str(example["holding_1"])

    query_embedding = embedder.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    scores, indices = index.search(query_embedding, k=5)

    current_passage = str(example["citing_prompt"]).strip().lower()

    best_passage = ""

    for idx in indices[0]:
        passage_text = knowledge_base_texts[int(idx)]
        passage_check = passage_text.strip().lower()

        # skip if retrieved passage contains the current passage
        if current_passage in passage_check:
            continue

        best_passage = passage_text
        break

    example["rag_passage"] = best_passage
    return example
    """

'\n#Embedding style\n!pip install -q sentence-transformers faiss-cpu\nfrom sentence_transformers import SentenceTransformer\nimport faiss\nimport numpy as np\n\nembedder = SentenceTransformer("all-MiniLM-L6-v2")\n\nknowledge_base_texts = [str(x) for x in knowledge_base]\n\nkb_embeddings = embedder.encode(\n    knowledge_base_texts,\n    convert_to_numpy=True,\n    normalize_embeddings=True,\n    show_progress_bar=True\n)\n\nindex = faiss.IndexFlatIP(kb_embeddings.shape[1])\nindex.add(kb_embeddings)\n\ndef add_rag_context(example):\n    query = str(example["citing_prompt"]) + " " + str(example["holding_1"])\n\n    query_embedding = embedder.encode(\n        [query],\n        convert_to_numpy=True,\n        normalize_embeddings=True\n    )\n\n    scores, indices = index.search(query_embedding, k=5)\n\n    current_passage = str(example["citing_prompt"]).strip().lower()\n\n    best_passage = ""\n\n    for idx in indices[0]:\n        passage_text = knowledge_base_texts[int(idx)]\n        pa

In [ ]:
small_train_dataset_rag = small_train_dataset.map(add_rag_context)
small_eval_dataset_rag = small_eval_dataset.map(add_rag_context)

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

In [ ]:
def tokenize_rag_function(examples):
    global header1, header2, header3, column_names1, column_names2, column_names3
    combined = [
        (
            header1 + (q if q is not None else "") + "" + (r if a is not None else ""),
            header2 + (p if p is not None else "") + header3 + (a if a is not None else "")
        )
        for q, p, a, r in zip(examples[column_names1], examples[column_names2], examples[column_names3], examples["rag_context"])
    ]

    return tokenizer(
        [c[0] for c in combined],
        [c[1] for c in combined],
        padding="max_length",
        truncation=True
    )



In [ ]:
def tokenize_rag_function_no_orginal_context(examples):
    global header1, header2, header3, column_names1, column_names2, column_names3
    combined = [
        (
            (r if a is not None else ""),
            header2 + (p if p is not None else "") + header3 + (a if a is not None else "")
        )
        for q, p, a, r in zip(examples[column_names1], examples[column_names2], examples[column_names3], examples["rag_context"])
    ]

    return tokenizer(
        [c[0] for c in combined],
        [c[1] for c in combined],
        padding="max_length",
        truncation=True
    )


In [ ]:
def tokenize_function_no_context(examples):
    global header1, header2, header3, column_names1, column_names2, column_names3
    combined = [
        (
            header2 + (p if p is not None else "") + header3 + (a if a is not None else "")
        )
        for p, a in zip(examples[column_names2], examples[column_names3])
    ]

    return tokenizer(
        [c[0] for c in combined],
        padding="max_length",
        truncation=True
    )

In [ ]:
def tokenize_function(examples):
    global header1, header2, header3, column_names1, column_names2, column_names3
    combined = [
        (
            header1 + (q if q is not None else ""),
            header2 + (p if p is not None else "") + header3 + (a if a is not None else "")
        )
        for q, p, a in zip(examples[column_names1], examples[column_names2], examples[column_names3])
    ]

    return tokenizer(
        [c[0] for c in combined],
        [c[1] for c in combined],
        padding="max_length",
        truncation=True
    )

In [ ]:
tokenized_train_dataset_rag = small_train_dataset_rag.map(tokenize_rag_function, batched=True)
tokenized_eval_dataset_rag = small_eval_dataset_rag.map(tokenize_rag_function, batched=True)

tokenized_train_dataset_rag_no_orginal_context = small_train_dataset_rag.map(tokenize_rag_function_no_orginal_context, batched=True)
tokenized_eval_dataset_rag_no_orginal_context = small_eval_dataset_rag.map(tokenize_rag_function_no_orginal_context, batched=True)


Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [ ]:
#no rag
tokenized_train_dataset_not_rag = small_train_dataset_rag.map(tokenize_function, batched=True)
tokenized_eval_dataset_not_rag = small_eval_dataset_rag.map(tokenize_function, batched=True)

tokenized_train_dataset_not_rag_no_context = small_train_dataset_rag.map(tokenize_function_no_context, batched=True)
tokenized_eval_dataset_not_rag_context = small_eval_dataset_rag.map(tokenize_function_no_context, batched=True)


Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [ ]:
#setting up the model

#default
model = AutoModelForSequenceClassification.from_pretrained("bert-base-cased", num_labels=2)

accuracy = evaluate.load("accuracy")
precision = evaluate.load("precision")
recall = evaluate.load("recall")
f1 = evaluate.load("f1")

#added average binary because it tells the model to Treat this as a binary classification problem and focus on the positive class (1)
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy.compute(predictions=predictions, references=labels)["accuracy"],
        "precision": precision.compute(predictions=predictions, references=labels, average="binary")["precision"],
        "recall": recall.compute(predictions=predictions, references=labels, average="binary")["recall"],
        "f1": f1.compute(predictions=predictions, references=labels, average="binary")["f1"],
    }


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-cased",
    num_labels=2
)

training_args = TrainingArguments(eval_strategy="epoch", num_train_epochs = 8, learning_rate = 2e-5)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset_rag,
    eval_dataset=tokenized_eval_dataset_rag,
    compute_metrics=compute_metrics
)
trainer.train()
trainer.evaluate()


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,0.714680,0.470000,0.000000,0.000000,0.000000
2,No log,0.636388,0.680000,0.652174,0.849057,0.737705
3,No log,0.658003,0.650000,0.666667,0.679245,0.672897
4,No log,0.674892,0.650000,0.666667,0.679245,0.672897
5,No log,0.726918,0.650000,0.680000,0.641509,0.660194
6,No log,0.823189,0.640000,0.639344,0.735849,0.684211
7,No log,0.854762,0.640000,0.660377,0.660377,0.660377
8,No log,0.884528,0.640000,0.673469,0.622642,0.647059


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'eval_loss': 0.8845279216766357,
 'eval_accuracy': 0.64,
 'eval_precision': 0.673469387755102,
 'eval_recall': 0.6226415094339622,
 'eval_f1': 0.6470588235294118,
 'eval_runtime': 0.745,
 'eval_samples_per_second': 134.227,
 'eval_steps_per_second': 17.45,
 'epoch': 8.0}

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-cased",
    num_labels=2
)

training_args = TrainingArguments(eval_strategy="epoch", num_train_epochs = 8, learning_rate =2e-5)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset_rag_no_orginal_context,
    eval_dataset=tokenized_eval_dataset_rag_no_orginal_context,
    compute_metrics=compute_metrics
)
trainer.train()
trainer.evaluate()


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,0.824955,0.580000,0.677419,0.396226,0.500000
2,No log,0.778894,0.610000,0.645833,0.584906,0.613861
3,No log,1.521139,0.540000,0.652174,0.283019,0.394737
4,No log,1.550375,0.600000,0.696970,0.433962,0.534884
5,No log,1.816157,0.550000,0.653846,0.320755,0.430380
6,No log,1.845306,0.610000,0.705882,0.452830,0.551724
7,No log,2.056222,0.590000,0.687500,0.415094,0.517647
8,No log,1.971990,0.600000,0.696970,0.433962,0.534884


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'eval_loss': 1.9719901084899902,
 'eval_accuracy': 0.6,
 'eval_precision': 0.696969696969697,
 'eval_recall': 0.4339622641509434,
 'eval_f1': 0.5348837209302325,
 'eval_runtime': 0.7569,
 'eval_samples_per_second': 132.115,
 'eval_steps_per_second': 17.175,
 'epoch': 8.0}

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-cased",
    num_labels=2
)

training_args = TrainingArguments(eval_strategy="epoch", num_train_epochs = 8, learning_rate =2e-5)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset_not_rag,
    eval_dataset=tokenized_eval_dataset_not_rag,
    compute_metrics=compute_metrics
)
trainer.train()
trainer.evaluate()


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,0.634357,0.680000,0.744186,0.603774,0.666667
2,No log,0.730649,0.640000,0.757576,0.471698,0.581395
3,No log,0.777830,0.740000,0.813953,0.660377,0.729167
4,No log,1.038374,0.700000,0.794872,0.584906,0.673913
5,No log,1.179201,0.700000,0.767442,0.622642,0.687500
6,No log,1.292559,0.700000,0.755556,0.641509,0.693878
7,No log,1.364720,0.700000,0.755556,0.641509,0.693878
8,No log,1.378279,0.670000,0.708333,0.641509,0.673267


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'eval_loss': 1.3782788515090942,
 'eval_accuracy': 0.67,
 'eval_precision': 0.7083333333333334,
 'eval_recall': 0.6415094339622641,
 'eval_f1': 0.6732673267326733,
 'eval_runtime': 0.7688,
 'eval_samples_per_second': 130.065,
 'eval_steps_per_second': 16.908,
 'epoch': 8.0}

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-cased",
    num_labels=2
)

training_args = TrainingArguments(eval_strategy="epoch", num_train_epochs = 8, learning_rate =2e-5)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset_not_rag_no_context,
    eval_dataset=tokenized_eval_dataset_not_rag_context,
    compute_metrics=compute_metrics
)
trainer.train()
trainer.evaluate()



Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,0.701024,0.470000,0.000000,0.000000,0.000000
2,No log,0.698636,0.470000,0.000000,0.000000,0.000000
3,No log,0.692230,0.530000,0.530000,1.000000,0.692810
4,No log,0.699524,0.470000,0.000000,0.000000,0.000000
5,No log,0.691508,0.530000,0.530000,1.000000,0.692810
6,No log,0.695406,0.470000,0.000000,0.000000,0.000000
7,No log,0.694296,0.470000,0.000000,0.000000,0.000000
8,No log,0.694045,0.470000,0.000000,0.000000,0.000000


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.p

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


{'eval_loss': 0.6940451860427856,
 'eval_accuracy': 0.47,
 'eval_precision': 0.0,
 'eval_recall': 0.0,
 'eval_f1': 0.0,
 'eval_runtime': 0.7551,
 'eval_samples_per_second': 132.433,
 'eval_steps_per_second': 17.216,
 'epoch': 8.0}